# Value registry — which categorical values exist, per field, per source

For SQL-type (Bucket-2) questions the user needs to know **what values a field can take** —
"cases per canton" is only askable if you know `canton` exists and holds `ZH/BE/AG/...`.
This notebook builds that registry automatically, read-only, for the three **raw imported
sources** and the **derived queryable tables**:

- a field counts as *categorical* if it has few distinct, short values (heuristic below);
- multi-value fields (ECHR `article` = `"8;8-1;13"`, list fields like `matched_keywords`)
  are **split and counted per element**;
- date-like fields are reported as a range + per-year counts;
- free text, IDs and URLs are listed as "not categorical" so their absence is explicit.

Output: `reports/value_registry.json` — `{source: {field: {values: {value: count}, ...}}}` —
plus a compact **SQL-hint block** per table (the value vocabulary an NL→SQL prompt or a
human query author can paste). **Topic-agnostic**: pure structure, no lexicons.

## 1. Sources, tables, and the categorical heuristic

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter
import pandas as pd

DATA_DIR, REPORT_DIR = Path("../data"), Path("../reports")
OUT = REPORT_DIR / "value_registry.json"

RAW_SOURCES = {
    "echr":  DATA_DIR / "echr_parental_alienation.json",
    "ris":   DATA_DIR / "ris_parental_alienation.json",
    "swiss": DATA_DIR / "swiss_parental_alienation.json",
}
DERIVED = {
    "echr_extracted (derived)": DATA_DIR / "echr_extracted.parquet",
    "echr_themes (derived)":    DATA_DIR / "echr_themes.parquet",
}
# any factory-deployed field tables join automatically
for pq in sorted(DATA_DIR.glob("field_*_deployed.parquet")):
    DERIVED[f"{pq.stem} (deployed field)"] = pq

MAX_DISTINCT   = 60      # more distinct values than this -> not categorical
MAX_VALUE_LEN  = 60      # longer values than this -> free text, not categorical
TOP_VALUES     = 40      # cap per field; the rest is bucketed as "(other)"
DATE_RE  = re.compile(r"^\d{4}-\d{2}-\d{2}|^\d{2}/\d{2}/\d{4}")
DATEISH  = re.compile(r"date|datum|geaendert|veroeffentlicht|scrape", re.I)
SKIP     = re.compile(r"full_text|content$|abstract|title|evidence|url|_id$|^id$|stable_id|"
                      r"itemid|ecli|appno|geschaeftszahl|docname|reference$|collection|"
                      r"source$|provenance|confidence|Signatur|Spider", re.I)
print("raw sources:", list(RAW_SOURCES), "| derived tables:", len(DERIVED))

raw sources: ['echr', 'ris', 'swiss'] | derived tables: 2


## 2. Build the registry

In [ ]:
def elements(v):
    """Split a value into countable elements (lists and ';'-joined strings)."""
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    s = str(v).strip()
    if ";" in s:
        return [p.strip() for p in s.split(";") if p.strip()]
    return [s]

def field_summary(values, n_rows):
    """values = list of non-empty raw values for one field."""
    non_empty = len(values)
    if not non_empty:
        return None
    sample = str(values[0])
    # date-like -> range + per-year counts
    if DATE_RE.match(sample):
        years = Counter()
        for v in values:
            m = re.search(r"(\d{4})", str(v))
            if m:
                years[m.group(1)] += 1
        return {"kind": "date", "coverage_pct": round(100 * non_empty / n_rows, 1),
                "range": [min(years), max(years)] if years else None,
                "per_year": dict(sorted(years.items()))}
    # element-wise counting
    c = Counter()
    for v in values:
        for e in elements(v):
            c[e] += 1
    if len(c) > MAX_DISTINCT or (c and max(len(k) for k in c) > MAX_VALUE_LEN):
        return {"kind": "not_categorical", "coverage_pct": round(100 * non_empty / n_rows, 1),
                "n_distinct": len(c), "example": str(values[0])[:80]}
    top = c.most_common(TOP_VALUES)
    out = {k: n for k, n in top}
    rest = sum(c.values()) - sum(n for _, n in top)
    if rest:
        out["(other)"] = rest
    return {"kind": "categorical", "coverage_pct": round(100 * non_empty / n_rows, 1),
            "n_distinct": len(c), "values": out}

registry = {}
for name, path in RAW_SOURCES.items():
    recs = json.loads(path.read_text())
    fields = sorted({k for r in recs for k in r})
    entry = {"rows": len(recs), "fields": {}}
    for f in fields:
        if SKIP.search(f):
            continue
        vals = [r[f] for r in recs if r.get(f) not in (None, "", [], {})]
        s = field_summary(vals, len(recs))
        if s:
            if s["kind"] != "date" and DATEISH.search(f) and s["kind"] == "not_categorical":
                continue                      # timestamps that failed the date regex: noise
            entry["fields"][f] = s
    registry[name] = entry
    n_cat = sum(1 for v in entry["fields"].values() if v["kind"] == "categorical")
    print(f"{name:8s}: {len(recs)} rows | categorical fields: {n_cat}")

for name, path in DERIVED.items():
    df = pd.read_parquet(path)
    entry = {"rows": len(df), "fields": {}}
    for f in df.columns:
        if SKIP.search(f):
            continue
        vals = [v for v in df[f].dropna().tolist() if v not in ("", [], {})]
        s = field_summary(vals, len(df))
        if s:
            entry["fields"][f] = s
    registry[name] = entry
    n_cat = sum(1 for v in entry["fields"].values() if v["kind"] == "categorical")
    print(f"{name:35s}: {len(df)} rows | categorical fields: {n_cat}")

OUT.write_text(json.dumps(registry, indent=2, ensure_ascii=False))
print(f"\nwrote {OUT} ({OUT.stat().st_size // 1024} KB)")

echr    : 1116 rows | categorical fields: 16
ris     : 548 rows | categorical fields: 11
swiss   : 2031 rows | categorical fields: 8
echr_extracted (derived)           : 1116 rows | categorical fields: 9
echr_themes (derived)              : 1116 rows | categorical fields: 20

wrote ../reports/value_registry.json (36 KB)


## 3. Human-readable view — the queryable vocabulary per source

In [ ]:
for name, entry in registry.items():
    cats = {f: v for f, v in entry["fields"].items() if v["kind"] == "categorical"}
    if not cats:
        continue
    print("=" * 72)
    print(f"{name}  ({entry['rows']} rows) — categorical fields you can query:")
    for f, v in sorted(cats.items(), key=lambda kv: kv[1]["n_distinct"]):
        vals = v["values"]
        preview = ", ".join(f"{k}({n})" for k, n in list(vals.items())[:8])
        more = f" ... +{v['n_distinct']-8} more" if v["n_distinct"] > 8 else ""
        print(f"  {f:22s} [{v['n_distinct']:3d} values, {v['coverage_pct']:5.1f}%] {preview}{more}")
    dates = {f: v for f, v in entry["fields"].items() if v["kind"] == "date"}
    for f, v in dates.items():
        print(f"  {f:22s} [date {v['range'][0]}..{v['range'][1]}, {v['coverage_pct']:5.1f}%]")

echr  (1116 rows) — categorical fields you can query:
  jurisdiction           [  1 values, 100.0%] Council of Europe(1116)
  lang                   [  1 values, 100.0%] en(1116)
  languageisocode        [  1 values, 100.0%] ENG(1116)
  separateopinion        [  2 values,  53.9%] FALSE(408), TRUE(194)
  doctype                [  3 values, 100.0%] HEJUD(567), HEDEC(311), HECOM(238)
  importance             [  4 values, 100.0%] 4(742), 3(267), 1(78), 2(29)
  matched_keywords       [  4 values, 100.0%] best interests of the child(748), contact rights(554), child welfare(227), parental alienation(41)
  doctypebranch          [  6 values, 100.0%] CHAMBER(450), COMMUNICATEDCASES(238), ADMISSIBILITY(167), ADMISSIBILITYCOM(144), COMMITTEE(78), GRANDCHAMBER(39)
  rulesofcourt           [  7 values,   3.7%] 17(25), 13(11), 16(5), 27(2), 32(1), 69(1), 93(1)
  typedescription        [  8 values, 100.0%] 15(536), 8(297), 23(238), 14(29), 9(10), 10(4), 18(1), 12(1)
  applicability          [  9 valu

## 4. SQL-hint block — paste-ready value vocabulary
A compact per-table string of field=values — usable by a human writing SQL **or appended to
the NL→SQL schema** so the local model maps entities ("Bern" → BE) without guessing.

In [ ]:
hints = {}
for name, entry in registry.items():
    parts = []
    for f, v in entry["fields"].items():
        if v["kind"] == "categorical" and v["n_distinct"] <= 25:
            vals = [k for k in v["values"] if k != "(other)"][:15]
            parts.append(f"{f} in [{', '.join(vals)}]")
    if parts:
        hints[name] = "; ".join(parts)
        print(f"--- {name} ---")
        print(hints[name][:400], "\n")
(REPORT_DIR / "value_registry_sql_hints.json").write_text(json.dumps(hints, indent=2, ensure_ascii=False))
print("wrote", REPORT_DIR / "value_registry_sql_hints.json")

--- echr ---
applicability in [55, 12, 36, 8, 41, 23, 20, 14, 21]; doctype in [HEJUD, HEDEC, HECOM]; doctypebranch in [CHAMBER, COMMUNICATEDCASES, ADMISSIBILITY, ADMISSIBILITYCOM, COMMITTEE, GRANDCHAMBER]; importance in [4, 3, 1, 2]; jurisdiction in [Council of Europe]; lang in [en]; languageisocode in [ENG]; matched_keywords in [best interests of the child, contact rights, child welfare, parental alienation]; 

--- ris ---
applikation in [Justiz]; dokumenttyp in [Text, Rechtssatz]; gericht in [OGH, AUSL EGMR]; jurisdiction in [AT]; lang in [de]; matched_keywords in [Kindeswohlgefährdung, Entfremdung, Loyalitätskonflikt]; organ in [OGH, AUSL EGMR]; rechtsgebiete in [Zivilrecht, Strafrecht, Undefined] 

--- swiss ---
Sprache in [de]; canton in [ZH, CH, BL, GR, BS, AG, BE, SO, FR, TG, SG, VS, NW, LU, SZ]; content_type in [application/pdf, text/html, charset=UTF-8]; jurisdiction in [CH]; lang in [de]; matched_keywords in [Loyalitätskonflikt, Kindeswohlgefährdung, Entfremdung, Kontaktverwe